In [3]:
# ============================================================
# GOOGLE DRIVE FILE COMPILER → SINGLE NUMBERED PDF
# Paste this entire block into ONE Google Colab cell.
# ============================================================

# Install required packages
!pip -q install pypdf reportlab pillow ipywidgets

import os
import io
import shutil
from pathlib import Path

from google.colab import drive
from IPython.display import display, clear_output
import ipywidgets as widgets

from PIL import Image
from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# 2. SETTINGS
# ============================================================

# CHANGE THIS EACH TIME YOU WANT TO USE A DIFFERENT FOLDER.
#
# Example:
# FOLDER = "/content/drive/MyDrive/My Documents"
#
# You can also use a subfolder:
# FOLDER = "/content/drive/MyDrive/Projects/2026/Reports"

FOLDER = "/content/drive/MyDrive/_Travel/Pre_Travel_SERVIR/EA_GSFC_26"


# Name of the final PDF
OUTPUT_FILENAME = "compiled_document.pdf"

# Supported files
PDF_EXTENSIONS = {".pdf"}
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff",
    ".bmp", ".webp"
}


In [6]:

# ============================================================
# 3. FIND FILES
# ============================================================

folder_path = Path(FOLDER)

if not folder_path.exists():
    raise FileNotFoundError(
        f"Folder does not exist:\n{FOLDER}\n\n"
        "Change the FOLDER variable near the top of the cell."
    )

files = []

for p in folder_path.iterdir():
    if p.is_file() and p.suffix.lower() in (
        PDF_EXTENSIONS | IMAGE_EXTENSIONS
    ):
        files.append(p)

files = sorted(files, key=lambda x: x.name.lower())

if not files:
    raise RuntimeError(
        f"No PDF or image files were found in:\n{FOLDER}"
    )


# ============================================================
# 4. BUILD FILE SELECTION UI
# ============================================================

print(f"Found {len(files)} PDF/image files.")
print(f"Folder: {folder_path}")
print()
print("Select the files you want to include.")
print("The selected files can then be reordered below.")


# Checkbox for every file
checkboxes = []

for i, file_path in enumerate(files):

    checkbox = widgets.Checkbox(
        value=False,
        description=file_path.name,
        indent=False,
        layout=widgets.Layout(width="95%")
    )

    checkboxes.append(checkbox)


# Put checkboxes into a scrollable area
file_box = widgets.VBox(
    checkboxes,
    layout=widgets.Layout(
        max_height="400px",
        overflow_y="auto",
        border="1px solid #ccc",
        padding="10px"
    )
)


# Select all / deselect all
select_all_button = widgets.Button(
    description="Select All",
    button_style="primary"
)

clear_all_button = widgets.Button(
    description="Clear All"
)


def select_all(_):
    for cb in checkboxes:
        cb.value = True


def clear_all(_):
    for cb in checkboxes:
        cb.value = False


select_all_button.on_click(select_all)
clear_all_button.on_click(clear_all)


display(
    widgets.HBox([
        select_all_button,
        clear_all_button
    ])
)

display(file_box)


# ============================================================
# 5. CREATE ORDERING UI
# ============================================================

order_output = widgets.Output()

move_up_button = widgets.Button(
    description="↑ Move Up",
    button_style="info"
)

move_down_button = widgets.Button(
    description="↓ Move Down",
    button_style="info"
)

remove_button = widgets.Button(
    description="Remove",
    button_style="warning"
)

compile_button = widgets.Button(
    description="COMPILE PDF",
    button_style="success",
    layout=widgets.Layout(
        width="200px",
        height="45px"
    )
)

status_output = widgets.Output()


# This will hold the selected files in their current order.
selected_files = []

# Currently highlighted item
selected_index = {"value": None}


def get_selected_from_checkboxes():

    return [
        files[i]
        for i, cb in enumerate(checkboxes)
        if cb.value
    ]


def refresh_order_list():

    with order_output:
        clear_output()

        selected_files.clear()
        selected_files.extend(get_selected_from_checkboxes())

        if not selected_files:
            print("No files selected.")
            return

        print("Current order:")
        print()

        for i, f in enumerate(selected_files, start=1):
            print(f"{i}. {f.name}")


def checkbox_changed(change):
    refresh_order_list()


for cb in checkboxes:
    cb.observe(checkbox_changed, names="value")


refresh_order_list()

display(widgets.HTML("<h3>Selected File Order</h3>"))
display(order_output)


# ============================================================
# 6. REORDERING
# ============================================================

# For simplicity and reliability in Colab, clicking a file
# number below selects it for moving.


order_selector = widgets.Dropdown(
    options=[],
    description="Select:",
    layout=widgets.Layout(width="600px")
)


def update_dropdown():

    options = []

    for i, f in enumerate(selected_files):
        options.append(
            (f"{i+1}. {f.name}", i)
        )

    order_selector.options = options


def refresh_all():

    refresh_order_list()
    update_dropdown()


def move_up(_):

    idx = order_selector.value

    if idx is None:
        return

    if idx > 0:

        selected_files[idx], selected_files[idx - 1] = (
            selected_files[idx - 1],
            selected_files[idx]
        )

        # Update checkbox state/order display only.
        with order_output:
            clear_output()
            print("Current order:")
            print()

            for i, f in enumerate(selected_files, start=1):
                print(f"{i}. {f.name}")

        update_dropdown()


def move_down(_):

    idx = order_selector.value

    if idx is None:
        return

    if idx < len(selected_files) - 1:

        selected_files[idx], selected_files[idx + 1] = (
            selected_files[idx + 1],
            selected_files[idx]
        )

        with order_output:
            clear_output()
            print("Current order:")
            print()

            for i, f in enumerate(selected_files, start=1):
                print(f"{i}. {f.name}")

        update_dropdown()


def remove_selected(_):

    idx = order_selector.value

    if idx is None:
        return

    if idx < len(selected_files):

        file_to_remove = selected_files[idx]

        # Uncheck corresponding original checkbox
        for i, f in enumerate(files):
            if f == file_to_remove:
                checkboxes[i].value = False
                break

        selected_files.pop(idx)

        with order_output:
            clear_output()

            if selected_files:
                print("Current order:")
                print()

                for i, f in enumerate(selected_files, start=1):
                    print(f"{i}. {f.name}")
            else:
                print("No files selected.")

        update_dropdown()


move_up_button.on_click(move_up)
move_down_button.on_click(move_down)
remove_button.on_click(remove_selected)

display(order_selector)

display(
    widgets.HBox([
        move_up_button,
        move_down_button,
        remove_button
    ])
)


# ============================================================
# 7. PDF CONVERSION FUNCTIONS
# ============================================================

def image_to_pdf(image_path):
    """
    Convert an image to a single-page PDF in memory.
    """

    img = Image.open(image_path)

    # Handle transparency
    if img.mode in ("RGBA", "LA", "P"):
        background = Image.new("RGB", img.size, "white")

        if img.mode == "P":
            img = img.convert("RGBA")

        background.paste(
            img,
            mask=img.getchannel("A")
            if "A" in img.getbands()
            else None
        )

        img = background

    else:
        img = img.convert("RGB")

    buffer = io.BytesIO()

    img.save(
        buffer,
        format="PDF",
        resolution=150.0
    )

    buffer.seek(0)

    return buffer


def create_page_numbers(input_pdf, output_pdf):
    """
    Add page numbers to every page.
    """

    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    total_pages = len(reader.pages)

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        # Determine page size
        width = float(page.mediabox.width)
        height = float(page.mediabox.height)

        # Create overlay
        overlay_buffer = io.BytesIO()

        c = canvas.Canvas(
            overlay_buffer,
            pagesize=(width, height)
        )

        # Page number
        text = f"{page_number} / {total_pages}"

        c.setFont("Helvetica", 9)

        # Bottom center
        c.drawCentredString(
            width / 2,
            18,
            text
        )

        c.save()

        overlay_buffer.seek(0)

        overlay_reader = PdfReader(overlay_buffer)
        overlay_page = overlay_reader.pages[0]

        page.merge_page(overlay_page)

        writer.add_page(page)

    with open(output_pdf, "wb") as f:
        writer.write(f)


# ============================================================
# 8. COMPILE EVERYTHING
# ============================================================

def compile_pdf(_):

    with status_output:
        clear_output()

        if not selected_files:
            print("ERROR: No files selected.")
            return

        print("Compiling...")
        print()

        # Temporary working directory
        temp_dir = Path("/content/pdf_compile_temp")

        if temp_dir.exists():
            shutil.rmtree(temp_dir)

        temp_dir.mkdir()

        combined_path = temp_dir / "combined.pdf"

        writer = PdfWriter()

        # ----------------------------------------------------
        # Process files in selected order
        # ----------------------------------------------------

        for position, file_path in enumerate(
            selected_files,
            start=1
        ):

            print(
                f"[{position}/{len(selected_files)}] "
                f"{file_path.name}"
            )

            suffix = file_path.suffix.lower()

            # -----------------------------------------------
            # PDF
            # -----------------------------------------------

            if suffix == ".pdf":

                reader = PdfReader(str(file_path))

                for page in reader.pages:
                    writer.add_page(page)

            # -----------------------------------------------
            # IMAGE
            # -----------------------------------------------

            elif suffix in IMAGE_EXTENSIONS:

                image_pdf = image_to_pdf(file_path)

                image_reader = PdfReader(image_pdf)

                for page in image_reader.pages:
                    writer.add_page(page)

        # ----------------------------------------------------
        # Write intermediate PDF
        # ----------------------------------------------------

        with open(combined_path, "wb") as f:
            writer.write(f)

        # ----------------------------------------------------
        # Add page numbers
        # ----------------------------------------------------

        output_path = folder_path / OUTPUT_FILENAME

        create_page_numbers(
            combined_path,
            output_path
        )

        # ----------------------------------------------------
        # Finished
        # ----------------------------------------------------

        print()
        print("=" * 60)
        print("DONE")
        print("=" * 60)
        print()
        print(f"Files compiled: {len(selected_files)}")
        print(f"Output: {output_path}")
        print()

        # Page count
        final_reader = PdfReader(str(output_path))

        print(
            f"Total pages: {len(final_reader.pages)}"
        )

        print()
        print("Your compiled PDF is saved in Google Drive.")


compile_button.on_click(compile_pdf)

display(widgets.HTML("<h3>Compile</h3>"))
display(compile_button)
display(status_output)


# ============================================================
# 9. INITIAL DROPDOWN UPDATE
# ============================================================

update_dropdown()

print()
print("Ready.")
print("1. Select your files above.")
print("2. Select a file in the ordering box.")
print("3. Use ↑ / ↓ to change its position.")
print("4. Click COMPILE PDF.")

Found 15 PDF/image files.
Folder: /content/drive/MyDrive/_Travel/Pre_Travel_SERVIR/EA_GSFC_26

Select the files you want to include.
The selected files can then be reordered below.


HTML(value='<h3>Selected File Order</h3>')

Output()

Dropdown(description='Select:', layout=Layout(width='600px'), options=(), value=None)

HTML(value='<h3>Compile</h3>')

Button(button_style='success', description='COMPILE PDF', layout=Layout(height='45px', width='200px'), style=B…

Output()


Ready.
1. Select your files above.
2. Select a file in the ordering box.
3. Use ↑ / ↓ to change its position.
4. Click COMPILE PDF.
